# 03c - Tester la branche GLiNER

Ce notebook ne lit pas de PDF. Il teste uniquement la branche `gliner` du pipeline.

`test_cases` contient seulement trois colonnes : `text`, `control_family`, `comment`.

Seule la colonne `text` est envoyee au traitement. `control_family` et `comment` servent uniquement a lire, filtrer et commenter les resultats dans le notebook.

Les domaines de reconnaissance GLiNER sont parametrables dans `GLINER_LABELS`.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
elif not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root


In [ ]:
from compliance_nlp import load_section_definitions
from compliance_nlp.pipeline import analyze_text

import pandas as pd


## Parametres GLiNER


In [ ]:
BRANCH_NAME = "gliner"
ENABLED_BRANCHES = (BRANCH_NAME,)
BRANCH_SCORE_COLUMN = "gliner_score"

GLINER_MODEL = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.50
GLINER_LABELS = [
    "donnee de sante",
    "opinion politique",
    "conviction religieuse",
    "appartenance syndicale",
    "orientation sexuelle",
    "origine ethnique",
    "donnee genetique",
    "donnee biometrique",
    "clause beneficiaire imprecise",
    "conseil non professionnel",
    "promesse de performance",
]

{
    "enabled_branches": ENABLED_BRANCHES,
    "gliner_model": GLINER_MODEL,
    "gliner_threshold": GLINER_THRESHOLD,
    "gliner_labels": GLINER_LABELS,
}


## Chargement des sections


In [ ]:
sections = load_section_definitions(repo_root / "configs" / "sections.csv")

pd.DataFrame([
    {"referentiel": "sections", "count": len(sections)},
    {"referentiel": "gliner_labels", "count": len(GLINER_LABELS)},
])


## Tableau des cas de test


In [ ]:
test_cases = pd.DataFrame([
    {
        "text": """
        7. Beneficiaires
        Je designe mes proches par parts egales.
        8. Declarations
        9. Conseil et Recommandation
        Texte neutre.
        10. Signatures
        """,
        "control_family": "generic",
        "comment": "Clause beneficiaire imprecise dans la section beneficiaires.",
    },
    {
        "text": """
        7. Beneficiaires
        Clause neutre.
        8. Declarations
        9. Conseil et Recommandation
        Ce placement est sans risque pour le client.
        10. Signatures
        """,
        "control_family": "generic",
        "comment": "Promesse ou minimisation de risque dans la section conseil.",
    },
    {
        "text": "Le client indique etre diabetique depuis plusieurs annees.",
        "control_family": "article9",
        "comment": "Donnee de sante explicite.",
    },
    {
        "text": "Observation libre : traitement par insuline signale par l'adherent.",
        "control_family": "article9",
        "comment": "Donnee de sante indirecte.",
    },
    {
        "text": "Le client est delegue syndical et participe a une campagne electorale.",
        "control_family": "article9",
        "comment": "Appartenance syndicale et opinion politique.",
    },
    {
        "text": "Le dossier mentionne un test genetique et une reconnaissance faciale.",
        "control_family": "article9",
        "comment": "Donnees genetiques et biometriques.",
    },
    {
        "text": "Le client souhaite modifier son adresse postale.",
        "control_family": "neutral",
        "comment": "Phrase neutre.",
    },
], columns=["text", "control_family", "comment"])

test_cases


## Fonctions d'execution


In [ ]:
def set_to_text(values):
    return " | ".join(sorted(value for value in values if value))


def run_detector(text, case_id):
    return analyze_text(
        document_name=case_id,
        source_path="notebook",
        extracted_text=text,
        generic_rules=[],
        section_definitions=sections,
        whitelist_terms=[],
        enabled_branches=ENABLED_BRANCHES,
        gliner_model=GLINER_MODEL,
        gliner_labels=GLINER_LABELS,
        gliner_threshold=GLINER_THRESHOLD,
    )


def findings_to_rows(case, analysis):
    findings = analysis.findings
    if not findings:
        return [{
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "comment": case["comment"],
            "detection_engine": BRANCH_NAME,
            "predicted_label": None,
            "matched_term": None,
            "section": None,
            "category": None,
            "detection_type": None,
            "score": None,
            "branch_score": None,
            "gliner_score": None,
            "evidence": None,
        }]

    return [
        {
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "comment": case["comment"],
            "detection_engine": finding.detection_engine,
            "predicted_label": finding.title,
            "matched_term": finding.matched_term,
            "section": finding.section,
            "category": finding.category,
            "detection_type": finding.detection_type,
            "score": finding.score,
            "branch_score": finding.branch_score,
            "gliner_score": finding.gliner_score,
            "evidence": finding.evidence,
        }
        for finding in findings
    ]


## Lancement du banc de test


In [ ]:
case_results = []
finding_rows = []

for case_index, case in test_cases.iterrows():
    case = case.copy()
    case["case_id"] = f"CASE_{case_index + 1:03d}"
    analysis = run_detector(case["text"], case["case_id"])
    findings = analysis.findings
    predicted_labels = {finding.title for finding in findings if finding.title}
    branch_scores = [getattr(finding, BRANCH_SCORE_COLUMN) for finding in findings]
    max_branch_score = max([score for score in branch_scores if score is not None], default=None)

    case_results.append({
        "case_id": case["case_id"],
        "text": case["text"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detected": bool(findings),
        "predicted_labels": set_to_text(predicted_labels),
        "finding_count": len(findings),
        "max_branch_score": max_branch_score,
        "branch_errors": analysis.metadata.get("branch_errors"),
    })
    finding_rows.extend(findings_to_rows(case, analysis))

case_results_df = pd.DataFrame(case_results)
findings_df = pd.DataFrame(finding_rows)

case_results_df


## Synthese de detection


In [ ]:
pd.DataFrame([{
    "branch": BRANCH_NAME,
    "cases": len(case_results_df),
    "detected_cases": int(case_results_df["detected"].sum()),
    "undetected_cases": int((~case_results_df["detected"]).sum()),
    "finding_count": int(case_results_df["finding_count"].sum()),
    "avg_max_branch_score": case_results_df["max_branch_score"].mean(),
    "max_branch_score": case_results_df["max_branch_score"].max(),
}])


## Cas sans detection


In [ ]:
case_results_df[~case_results_df["detected"]][[
    "case_id",
    "control_family",
    "comment",
    "text",
    "branch_errors",
]]


## Detail des findings


In [ ]:
findings_df.sort_values(["case_id", "branch_score"], ascending=[True, False], na_position="last")


## Scores par label


In [ ]:
findings_df[findings_df["predicted_label"].notna()].groupby(
    ["detection_engine", "predicted_label", "section"], dropna=False
).agg(
    count=("predicted_label", "count"),
    avg_score=("branch_score", "mean"),
    min_score=("branch_score", "min"),
    max_score=("branch_score", "max"),
).reset_index().sort_values(["detection_engine", "predicted_label", "section"])
